# 04 — Gold: ML feature store

Fits and persists the feature pipeline (StringIndexer → OneHotEncoder → VectorAssembler
→ StandardScaler), writes the Gold feature-vector table, stores the pipeline model in the
artifacts volume so scoring reuses the exact same encoding, and writes a feature manifest
that records what every slot of the assembled vector actually is.

Target: `label = 1` if `arrival_delay >= 15` else `0`. Cancelled / diverted (null delay)
is filtered here.


In [ ]:
import sys
sys.path.append("..")

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler,
)
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

from src import config
from src.feature_space import expand_feature_space

silver = spark.table(config.SILVER).filter(col("arrival_delay").isNotNull())
labeled = silver.withColumn(
    "label",
    when(col("arrival_delay") >= config.DELAY_THRESHOLD_MINUTES, 1.0).otherwise(0.0),
)

print(f"Silver rows (non-cancelled): {labeled.count():,}")
positive_rate = labeled.filter(col("label") == 1.0).count() / labeled.count()
print(f"Positive class rate: {positive_rate:.3%}")


## Feature groups

In [ ]:
categorical_cols = [
    "airline_name", "airline_code", "origin_airport_code",
    "destination_airport_code", "season",
]
boolean_cols = ["is_weekend", "is_holiday", "is_near_holiday", "is_holiday_period"]
numerical_cols = [
    "flight_month", "flight_year", "day_of_week", "week_of_year", "day_of_month",
    "quarter", "fl_number", "crs_elapsed_time", "distance", "dep_hour", "arr_hour",
    "dep_delay",
]

# Defined unconditionally, at module level of the notebook. It used to live inside
# the `except NameError:` branch below, which meant that on any second run in the
# same session the reuse path skipped it and the manifest cell died with NameError.
assembled_cols = (
    numerical_cols + boolean_cols + [f"{c}_ohe" for c in categorical_cols]
)
print(f"{len(assembled_cols)} assembled input columns "
      f"({len(numerical_cols)} numeric, {len(boolean_cols)} boolean, "
      f"{len(categorical_cols)} one-hot blocks)")


## Fit or reuse the feature pipeline — explicitly

The previous version decided whether to refit by catching `NameError` on a variable
left over from an earlier execution:

```python
try:
    _ = pipeline_model          # survives from a previous run?
except NameError:
    ...                         # no: fit one
```

That makes behaviour depend on hidden session state. The same notebook, on the same
inputs, takes two different code paths depending on whether someone ran a cell
earlier — which is the definition of not reproducible, and the first thing a reviewer
notices. It also silently skipped the definition of `assembled_cols`, so the reuse
path reliably crashed two cells later.

The constraint it was working around is real: fitting a `StringIndexer` over
high-cardinality airport codes is expensive and the serverless ML cache is capped at
1 GB. The answer is to cache the artifact where artifacts belong — the volume — and
make the choice explicit and inspectable:

- **`auto`** (default) — load the saved pipeline if it exists, otherwise fit and save
- **`always`** — refit and overwrite, for when Silver's schema changed
- **`never`** — load only, and fail loudly if there is nothing to load

Same inputs plus same widget now means same path, every time, in any session.


In [ ]:
dbutils.widgets.dropdown("refit_pipeline", "auto", ["auto", "always", "never"])
REFIT = dbutils.widgets.get("refit_pipeline")

pipeline_path = f"{config.ARTIFACT_VOLUME}/feature_pipeline"


def _saved_pipeline_exists() -> bool:
    try:
        dbutils.fs.ls(pipeline_path)
        return True
    except Exception:
        return False


def _build_pipeline() -> Pipeline:
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in categorical_cols
    ]
    encoders = [
        OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
        for c in categorical_cols
    ]
    assembler = VectorAssembler(
        inputCols=assembled_cols, outputCol="features_raw", handleInvalid="keep",
    )
    scaler = StandardScaler(
        inputCol="features_raw", outputCol="features", withMean=False, withStd=True,
    )
    return Pipeline(stages=[*indexers, *encoders, assembler, scaler])


# Null handling is identical on both paths, so it lives outside the branch.
prepared = labeled.na.fill(0, subset=numerical_cols + boolean_cols)
for c in categorical_cols:
    prepared = prepared.na.fill("UNKNOWN", subset=[c])

exists = _saved_pipeline_exists()
if REFIT == "never" and not exists:
    raise FileNotFoundError(
        f"refit_pipeline='never' but no pipeline at {pipeline_path}. "
        "Run once with 'auto' or 'always' first."
    )

if REFIT == "always" or (REFIT == "auto" and not exists):
    print(f"Fitting a new feature pipeline (refit_pipeline={REFIT}, exists={exists})")
    pipeline_model = _build_pipeline().fit(prepared)
    pipeline_model.write().overwrite().save(pipeline_path)
    print(f"Saved to {pipeline_path}")
else:
    print(f"Loading the saved pipeline (refit_pipeline={REFIT})")
    pipeline_model = PipelineModel.load(pipeline_path)

gold = pipeline_model.transform(prepared).select(
    "label", "features", *[c for c in labeled.columns if c != "label"],
)
print(f"Gold DataFrame: {len(gold.columns)} columns")


## Persist Gold

In [ ]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.GOLD)
)
gold_count = spark.table(config.GOLD).count()
print(f"Gold rows: {gold_count:,}")


In [ ]:
spark.sql(f"""
    ALTER TABLE {config.GOLD} SET TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact  = true
    )
""")

for name, expr in {
    "label_is_binary": "label IN (0.0, 1.0)",
    "features_present": "features IS NOT NULL",
    "flight_year_present": "flight_year IS NOT NULL",
}.items():
    spark.sql(f"ALTER TABLE {config.GOLD} DROP CONSTRAINT IF EXISTS {name}")
    spark.sql(f"ALTER TABLE {config.GOLD} ADD CONSTRAINT {name} CHECK ({expr})")
    print(f"  constraint {name:<22} {expr}")

# 05_train filters this table on flight_year three times — once per window.
spark.sql(f"OPTIMIZE {config.GOLD} ZORDER BY (flight_year)")
print(f"\nOPTIMIZE ... ZORDER BY (flight_year) — the column every training split filters on")


## Feature manifest — real vector indices

The manifest used to record `position = i` over `assembled_cols`. That is the position
of a *column* in the assembler's input list, not the index of a *slot* in the output
vector, and the two stop agreeing the moment a one-hot block appears: five categorical
columns expand into hundreds of slots. `position` was therefore correct for the twelve
numerics at the front and quietly wrong for everything after them (C4).

`src.feature_space.expand_feature_space` walks the fitted pipeline and expands each
one-hot block to its true width, so every row here is one vector slot.

**Why not read the vector's own `ml_attr` metadata?** Because it isn't there. The
pipeline ends in a `StandardScaler`, which does not propagate its input's attribute
names, and the vector then round-trips through a Delta write. The first full run
confirmed it: `05_train` reported names resolved from the saved pipeline, not from
metadata. So the manifest is not a convenience — it is the only durable record of what
the feature vector contains, and `05_train` now reads it instead of re-deriving it.


In [ ]:
manifest_rows = expand_feature_space(pipeline_model)
manifest_df = spark.createDataFrame(manifest_rows)

(
    manifest_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.FEATURE_MANIFEST)
)

vector_width = len(manifest_rows)
print(f"Feature manifest: {vector_width:,} rows -> {config.FEATURE_MANIFEST}")

# The manifest is only useful if it describes the vector that actually exists.
first_vec = spark.table(config.GOLD).select("features").first()["features"]
print(f"Manifest slots: {vector_width:,}   actual vector width: {len(first_vec):,}")
assert vector_width == len(first_vec), (
    f"manifest describes {vector_width} slots but the vector has {len(first_vec)}"
)
print("Manifest width matches the assembled vector.")

display(manifest_df.orderBy("vector_index").limit(20))


In [ ]:
# dep_delay is the column 05_train must exclude for the pre-departure variant.
# Resolving it here is a cheap check that the manifest is usable for that purpose.
dep = manifest_df.filter(col("name") == "dep_delay").first()
assert dep is not None, "dep_delay missing from the manifest"
print(f"dep_delay -> vector_index {dep['vector_index']}")

display(
    manifest_df.groupBy("source_column", "attr_type")
    .agg(F.count("*").alias("vector_slots"))
    .orderBy(F.desc("vector_slots"))
    .limit(15)
)
print("One-hot blocks are where position and vector index diverge — the widest")
print("block above occupies that many slots on its own.")


In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {config.GOLD}").select(
    "version", "timestamp", "operation", "operationMetrics"
).limit(10))
